# VQ-VAE

In [1]:
"""
VQ‑VAE (Vector‑Quantized Variational Autoencoder) — Minimal PyTorch Demo

This script:
- Implements a VQ layer with EMA updates (VectorQuantizerEMA)
- Builds a tiny Conv encoder/decoder for 32x32 grayscale images
- Shows a toy training loop with dummy data (replace with a real DataLoader)

Run: python vqvae_demo.py

Notes:
- For RGB, change in_ch=3.
- For other image sizes, adjust the encoder/decoder strides or add blocks so that
  downsampling/upsampling are consistent.
"""
from dataclasses import dataclass
from typing import Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:

# -------------------------
# Vector Quantizer (EMA)
# -------------------------
class VectorQuantizerEMA(nn.Module):
    """EMA‑based codebook as in VQ‑VAE v2.

    Args:
        n_e: number of embeddings (codebook entries)
        e_dim: embedding dimension
        decay: EMA decay for codebook updates (typ. 0.99–0.999)
        eps: small value to avoid div‑by‑zero
        commitment_cost: beta in loss, encourages encoder outputs to stay close to codes
    """
    def __init__(self, n_e: int, e_dim: int, decay: float = 0.99, eps: float = 1e-5, commitment_cost: float = 0.25):
        super().__init__()
        self.n_e = n_e
        self.e_dim = e_dim
        self.decay = decay
        self.eps = eps
        self.commitment_cost = commitment_cost

        embed = torch.randn(n_e, e_dim)
        self.register_buffer("embedding", embed)
        self.register_buffer("cluster_size", torch.zeros(n_e))
        self.register_buffer("embed_avg", embed.clone())

    @torch.no_grad()
    def _ema_update(self, z_e_flat: torch.Tensor, codes: torch.Tensor):
        # z_e_flat: [B*H*W, D]
        # codes: [B*H*W] with indices in [0, n_e)
        one_hot = F.one_hot(codes, num_classes=self.n_e).type_as(z_e_flat)
        cluster_size = one_hot.sum(dim=0)
        embed_sum = one_hot.t() @ z_e_flat

        self.cluster_size.mul_(self.decay).add_(cluster_size, alpha=1 - self.decay)
        self.embed_avg.mul_(self.decay).add_(embed_sum, alpha=1 - self.decay)

        # Laplace smoothing to avoid zeros
        n = self.cluster_size.sum()
        cluster_size = (self.cluster_size + self.eps) / (n + self.n_e * self.eps) * n

        embed_normalized = self.embed_avg / cluster_size.unsqueeze(1)
        self.embedding.copy_(embed_normalized)

    def forward(self, z_e: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        z_e: [B, D, H, W] — encoder outputs
        Returns:
            z_q: quantized tensor [B, D, H, W]
            aux: dict with losses and stats (loss, perplexity, codes)
        """
        B, D, H, W = z_e.shape
        z_flat = z_e.permute(0, 2, 3, 1).contiguous().view(-1, D)  # [BHW, D]

        # Compute L2 distance to each embedding: ||z - e||^2 = z^2 + e^2 - 2 z·e
        # Efficiently: use dot product and precompute norms
        embedding = self.embedding
        z_sq = (z_flat ** 2).sum(1, keepdim=True)  # [BHW, 1]
        e_sq = (embedding ** 2).sum(1)  # [n_e]
        # distances: [BHW, n_e]
        distances = z_sq + e_sq.unsqueeze(0) - 2 * (z_flat @ embedding.t())

        codes = torch.argmin(distances, dim=1)  # [BHW]
        z_q_flat = embedding.index_select(0, codes)  # [BHW, D]
        z_q = z_q_flat.view(B, H, W, D).permute(0, 3, 1, 2).contiguous()

        # Straight‑through estimator: replace gradients for encoder
        z_q_st = z_e + (z_q - z_e).detach()

        # Loss terms
        # commitment: ||z_e.detach() - z_q||^2; codebook is updated via EMA, so no codebook loss
        commitment_loss = self.commitment_cost * F.mse_loss(z_e, z_q.detach())
        loss = commitment_loss

        # Perplexity (how many codes used)
        with torch.no_grad():
            encodings = F.one_hot(codes, num_classes=self.n_e).float()
            avg_probs = encodings.mean(dim=0)
            perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        # EMA codebook update (no grad)
        self._ema_update(z_flat.detach(), codes.detach())

        aux = {
            "loss": loss,
            "perplexity": perplexity,
            "codes": codes.view(B, H, W)
        }
        return z_q_st, aux

# -------------------------
# Encoder / Decoder
# -------------------------
class Encoder(nn.Module):
    def __init__(self, in_ch: int, hidden: int, z_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 4, stride=2, padding=1),  # 32->16
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 4, stride=2, padding=1),  # 16->8
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, z_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self, out_ch: int, hidden: int, z_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(z_dim, hidden, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, hidden, 4, stride=2, padding=1),  # 8->16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(hidden, out_ch, 4, stride=2, padding=1),  # 16->32
        )

    def forward(self, z_q):
        return self.net(z_q)

# -------------------------
# VQ‑VAE wrapper
# -------------------------
@dataclass
class VQVAEConfig:
    in_ch: int = 1          # 1 for grayscale, 3 for RGB
    hidden: int = 128
    z_dim: int = 64         # embedding dim D
    n_embed: int = 512      # codebook size K
    decay: float = 0.99
    commitment_cost: float = 0.25

class VQVAE(nn.Module):
    def __init__(self, cfg: VQVAEConfig):
        super().__init__()
        self.enc = Encoder(cfg.in_ch, cfg.hidden, cfg.z_dim)
        self.vq = VectorQuantizerEMA(cfg.n_embed, cfg.z_dim, cfg.decay, commitment_cost=cfg.commitment_cost)
        self.dec = Decoder(cfg.in_ch, cfg.hidden, cfg.z_dim)

    def forward(self, x) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        z_e = self.enc(x)
        z_q, aux = self.vq(z_e)
        x_recon = self.dec(z_q)
        return x_recon, aux

# -------------------------
# Training demo
# -------------------------

def reconstruction_loss(x, x_recon):
    # For images in [-1,1] use MSE or L1; for [0,1] + Bernoulli model use BCE.
    return F.mse_loss(x_recon, x)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    cfg = VQVAEConfig(in_ch=1, hidden=128, z_dim=64, n_embed=512, decay=0.99, commitment_cost=0.25)
    model = VQVAE(cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)

    # --- Dummy data (B, C, H, W) = (64, 1, 32, 32). Replace with a real DataLoader.
    def batch():
        x = torch.randn(64, cfg.in_ch, 32, 32)  # values ~N(0,1)
        return x.clamp(-1, 1)

    model.train()
    for step in range(1, 501):
        x = batch().to(device)
        x_recon, aux = model(x)
        recon = reconstruction_loss(x, x_recon)
        vq_loss = aux["loss"]
        loss = recon + vq_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        if step % 50 == 0:
            with torch.no_grad():
                psnr = 10 * torch.log10(1.0 / recon.clamp_min(1e-8))
            print(f"step {step:04d} | loss {loss.item():.4f} | recon {recon.item():.4f} | vq {vq_loss.item():.4f} | perplex {aux['perplexity'].item():.1f} | PSNR {psnr.item():.2f}")

    # Example: encode→quantize→decode on a single batch
    model.eval()
    with torch.no_grad():
        x = batch().to(device)
        x_recon, aux = model(x)
        print("Final perplexity:", aux["perplexity"].item())


In [3]:
main()

step 0050 | loss 0.5299 | recon 0.5163 | vq 0.0135 | perplex 1.0 | PSNR 2.87
step 0100 | loss 0.6024 | recon 0.5159 | vq 0.0865 | perplex 1.0 | PSNR 2.87
step 0150 | loss 0.6209 | recon 0.5178 | vq 0.1031 | perplex 1.0 | PSNR 2.86
step 0200 | loss 0.6131 | recon 0.5153 | vq 0.0978 | perplex 1.0 | PSNR 2.88
step 0250 | loss 0.6110 | recon 0.5153 | vq 0.0957 | perplex 1.0 | PSNR 2.88
step 0300 | loss 0.6106 | recon 0.5181 | vq 0.0925 | perplex 1.0 | PSNR 2.86
step 0350 | loss 0.5932 | recon 0.5141 | vq 0.0791 | perplex 1.0 | PSNR 2.89
step 0400 | loss 0.5868 | recon 0.5152 | vq 0.0715 | perplex 1.0 | PSNR 2.88
step 0450 | loss 0.5776 | recon 0.5160 | vq 0.0616 | perplex 1.0 | PSNR 2.87
step 0500 | loss 0.5677 | recon 0.5177 | vq 0.0500 | perplex 1.0 | PSNR 2.86
Final perplexity: 1.0
